In [ ]:
pip install requests beautifulsoup4 langchain-text-splitters sentence-transformers faiss-cpu transformers peft torch

In [4]:
pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.5 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import torch
import requests
import numpy as np
import faiss
from bs4 import BeautifulSoup
from typing import List, Deque
from collections import deque
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ==========================================
# 1. SCRAPER
# ==========================================
class SerenityScraper:
    def __init__(self):
        self.urls = [
            "https://en.wikipedia.org/wiki/Empathy",
            "https://en.wikipedia.org/wiki/Emotional_intelligence",
            "https://en.wikipedia.org/wiki/Active_listening",
            "https://www.verywellmind.com/what-is-coping-4842146"
        ]
        self.headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=700,
            chunk_overlap=100,
            separators=["\n\n", "\n", ".", " "]
        )

    def fetch_and_process(self) -> List[str]:
        chunks = []
        print("🌐 Scraping public knowledge sources...")
        for url in self.urls:
            try:
                r = requests.get(url, headers=self.headers, timeout=10)
                if r.status_code != 200:
                    continue
                soup = BeautifulSoup(r.text, "html.parser")
                for tag in soup(["script", "style", "nav", "header", "footer"]):
                    tag.decompose()
                text = " ".join([p.get_text() for p in soup.find_all('p')])
                if len(text.split()) < 40:
                    continue
                chunks.extend(self.splitter.split_text(text))
            except Exception as e:
                print(f"⚠️ Could not scrape {url}: {e}")
        print(f"✅ Processed {len(chunks)} knowledge chunks.")
        return chunks

# ==========================================
# 2. VECTOR DB
# ==========================================
class SerenityVectorDB:
    def __init__(self, chunks: List[str]):
        self.chunks = chunks
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.dimension = self.embed_model.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatL2(self.dimension)

    def build(self):
        print("🧠 Creating vector embeddings...")
        embeddings = self.embed_model.encode(self.chunks, show_progress_bar=True)
        self.index.add(np.array(embeddings).astype("float32"))
        print("✅ FAISS index built.")

    def query(self, text: str, k: int = 3) -> List[str]:
        query_vec = self.embed_model.encode([text]).astype("float32")
        _, indices = self.index.search(query_vec, k)
        return [self.chunks[i] for i in indices[0] if i != -1]

# ==========================================
# 3. LLM
# ==========================================
class SerenityLLM:
    def __init__(self, model_id="Qwen/Qwen2.5-1.5B-Instruct", lora_path=None):
        print(f"🤖 Loading LLM: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", torch_dtype="auto"
        )
        if lora_path and os.path.exists(lora_path):
            print(f"✨ Applying LoRA adapter from {lora_path}...")
            self.model = PeftModel.from_pretrained(self.model, lora_path)

    def generate(self, user_input: str, context_chunks: List[str], memory: Deque[str] = None) -> str:
        context_text = "\n".join([f"- {c}" for c in context_chunks])
        memory_text = "\n".join(memory) if memory else ""

        prompt = (
            "You are Serenity, a warm and empathetic therapist.\n"
            "Reflect feelings, validate emotions, and ask at most ONE follow-up question.\n"
            "Keep responses concise (1-3 sentences), human-like, and direct.\n"
            f"Context:\n{context_text}\n"
            f"Conversation memory:\n{memory_text}\n" if memory_text else ""
            f"User: {user_input}\n"
            "Assistant:"
        )

        inputs = self.tokenizer([prompt], return_tensors="pt").to(self.model.device)
        generated_ids = self.model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            eos_token_id=self.tokenizer.eos_token_id
        )
        response = self.tokenizer.batch_decode(
            generated_ids[:, inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )[0]
        return response.strip()

# ==========================================
# 4. MAIN ORCHESTRATION
# ==========================================
def run_serenity():
    # Step 1: Scraper
    scraper = SerenityScraper()
    knowledge_chunks = scraper.fetch_and_process()
    if not knowledge_chunks:
        print("❌ No knowledge collected. Exiting.")
        return

    # Step 2: FAISS vector index
    vector_db = SerenityVectorDB(knowledge_chunks)
    vector_db.build()

    # Step 3: Load LLM
    serenity_brain = SerenityLLM(lora_path=None)  # add path if you have LoRA

    # Step 4: Memory buffer (last 2 user inputs)
    memory_buffer: Deque[str] = deque(maxlen=2)

    print("\n" + "="*50)
    print("🌿 SERENITY: Empathetic Assistant Online")
    print("Type 'quit' to exit.")
    print("="*50 + "\n")

    while True:
        user_msg = input("You: ")
        if user_msg.lower() in ["quit", "exit", "bye"]:
            print("\nSerenity: I'm here whenever you need to talk again. Take care. 🌿")
            break

        # Retrieve context
        context = vector_db.query(user_msg, k=3)

        # Generate response
        response = serenity_brain.generate(user_msg, context, memory=memory_buffer)

        # Print and update memory
        print(f"\nSerenity: {response}\n")
        memory_buffer.append(f"User: {user_msg}")
        memory_buffer.append(f"Assistant: {response}")

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    run_serenity()


In [ ]:
"""
SERENITY - RAG Therapist (FIXED for Kaggle)
Copy this ENTIRE file into ONE cell and run it
"""

# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================
!pip install -q beautifulsoup4 requests sentence-transformers faiss-cpu transformers accelerate

# =============================================================================
# IMPORTS
# =============================================================================
import requests
from bs4 import BeautifulSoup
import time
import json
import os
from typing import List, Dict, Tuple
from urllib.parse import urlparse
import re
from dataclasses import dataclass, asdict
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =============================================================================
# CLASSES
# =============================================================================

@dataclass
class Chunk:
    chunk_id: str
    text: str
    source_title: str
    source_url: str
    source_type: str
    word_count: int
    chunk_index: int
    def to_dict(self) -> Dict:
        return asdict(self)

class KnowledgeScraper:
    def __init__(self, output_dir: str = "knowledge_base"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        
    def scrape_wikipedia_article(self, url: str) -> Dict:
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            title = soup.find('h1', class_='firstHeading')
            title_text = title.get_text() if title else "Unknown"
            content_div = soup.find('div', class_='mw-parser-output')
            if not content_div:
                return None
            paragraphs = []
            for p in content_div.find_all('p'):
                text = p.get_text().strip()
                if len(text) > 40 and not text.startswith('['):
                    text = re.sub(r'\[\d+\]', '', text)
                    paragraphs.append(text)
            content = '\n\n'.join(paragraphs)
            return {'title': title_text, 'url': url, 'content': content, 'source': 'Wikipedia', 'word_count': len(content.split())}
        except Exception as e:
            print(f"Error scraping {url}: {str(e)}")
            return None
    
    def scrape_urls(self, urls: List[str], delay: float = 1.0) -> List[Dict]:
        articles = []
        for i, url in enumerate(urls):
            print(f"Scraping {i+1}/{len(urls)}: {url}")
            article = self.scrape_wikipedia_article(url)
            if article and article['word_count'] > 50:
                articles.append(article)
                print(f"  ✓ {article['title']} ({article['word_count']} words)")
            if i < len(urls) - 1:
                time.sleep(delay)
        return articles

class TextProcessor:
    def __init__(self, chunk_size: int = 180, overlap: int = 30):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def clean_text(self, text: str) -> str:
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\w\s.,!?;:\-\'"()]', '', text)
        return text.strip()
    
    def split_into_sentences(self, text: str) -> List[str]:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sentences if s.strip()]
    
    def create_chunks(self, text: str, min_words: int = 40) -> List[str]:
        sentences = self.split_into_sentences(text)
        chunks = []
        current_chunk = []
        current_word_count = 0
        for sentence in sentences:
            sentence_word_count = len(sentence.split())
            if current_word_count + sentence_word_count > self.chunk_size and current_chunk:
                chunks.append(' '.join(current_chunk))
                overlap_sentences = []
                overlap_words = 0
                for sent in reversed(current_chunk):
                    if overlap_words + len(sent.split()) <= self.overlap:
                        overlap_sentences.insert(0, sent)
                        overlap_words += len(sent.split())
                    else:
                        break
                current_chunk = overlap_sentences
                current_word_count = overlap_words
            current_chunk.append(sentence)
            current_word_count += sentence_word_count
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        return chunks
    
    def process_article(self, article: Dict, article_idx: int) -> List[Chunk]:
        cleaned_content = self.clean_text(article['content'])
        chunk_texts = self.create_chunks(cleaned_content)
        chunks = []
        for chunk_idx, chunk_text in enumerate(chunk_texts):
            chunk = Chunk(
                chunk_id=f"article_{article_idx}_chunk_{chunk_idx}",
                text=chunk_text,
                source_title=article['title'],
                source_url=article['url'],
                source_type=article['source'],
                word_count=len(chunk_text.split()),
                chunk_index=chunk_idx
            )
            chunks.append(chunk)
        return chunks
    
    def process_articles(self, articles: List[Dict]) -> List[Chunk]:
        all_chunks = []
        for idx, article in enumerate(articles):
            chunks = self.process_article(article, idx)
            all_chunks.extend(chunks)
            print(f"Processed: {article['title']} -> {len(chunks)} chunks")
        return all_chunks

class EmbeddingEngine:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        self.index = None
        self.chunks = []
        
    def build_index(self, chunks: List[Chunk]):
        print(f"Building FAISS index for {len(chunks)} chunks...")
        self.chunks = chunks
        chunk_texts = [chunk.text for chunk in chunks]
        embeddings = self.model.encode(chunk_texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
        faiss.normalize_L2(embeddings)
        self.index = faiss.IndexFlatIP(self.embedding_dim)
        self.index.add(embeddings.astype('float32'))
        print(f"✓ FAISS index built with {self.index.ntotal} vectors")
    
    def search(self, query: str, top_k: int = 3) -> List[Tuple[Chunk, float]]:
        query_embedding = self.model.encode([query], show_progress_bar=False, convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)
        similarities, indices = self.index.search(query_embedding.astype('float32'), min(top_k, self.index.ntotal))
        results = []
        for idx, score in zip(indices[0], similarities[0]):
            if idx < len(self.chunks):
                results.append((self.chunks[idx], float(score)))
        return results
    
    def get_context(self, query: str, top_k: int = 3, max_words: int = 500) -> Tuple[str, List[Dict]]:
        results = self.search(query, top_k)
        context_parts = []
        metadata = []
        total_words = 0
        for i, (chunk, score) in enumerate(results):
            if total_words + chunk.word_count > max_words and i > 0:
                break
            context_parts.append(f"[Source {i+1}]: {chunk.text}")
            total_words += chunk.word_count
            metadata.append({'source': chunk.source_title, 'url': chunk.source_url, 'similarity': score, 'chunk_id': chunk.chunk_id})
        return "\n\n".join(context_parts), metadata

class SerenityGenerator:
    def __init__(self, base_model: str = "Qwen/Qwen2.5-1.5B-Instruct", embedding_engine=None):
        print(f"Loading model: {base_model}...")
        self.model = AutoModelForCausalLM.from_pretrained(
            base_model, 
            device_map="auto", 
            trust_remote_code=True, 
            torch_dtype=torch.float16
        )
        self.tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.eval()
        self.embedding_engine = embedding_engine
        self.conversation_history = []
        print("✓ Serenity loaded!")
    
    def get_system_prompt(self) -> str:
        return """You are Serenity, a warm and empathetic therapist assistant. Your role is to:
- Listen actively and validate the user's feelings
- Show genuine empathy and compassion
- Reflect back what you hear to demonstrate understanding
- Provide gentle, supportive guidance based on therapeutic best practices
- Use the retrieved context naturally to inform your responses
- Keep responses conversational, warm, and human-like (150-250 words)
- Ask thoughtful follow-up questions when appropriate

Remember: You are here to support, not to diagnose or replace professional therapy."""
    
    def generate_response(self, user_message: str, use_rag: bool = True, top_k_chunks: int = 3, max_new_tokens: int = 250, temperature: float = 0.7, top_p: float = 0.9) -> Tuple[str, List[Dict]]:
        # Get context
        context = ""
        rag_metadata = []
        if use_rag and self.embedding_engine:
            context, rag_metadata = self.embedding_engine.get_context(user_message, top_k=top_k_chunks, max_words=400)
        
        # Build messages
        messages = []
        system_content = self.get_system_prompt()
        if context:
            system_content += f"\n\nRelevant knowledge:\n{context}"
        messages.append({"role": "system", "content": system_content})
        
        # Add history
        if self.conversation_history:
            for msg in self.conversation_history[-6:]:
                messages.append(msg)
        
        # Add current message
        messages.append({"role": "user", "content": user_message})
        
        # Generate
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id
        )
        
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Update history
        self.conversation_history.append({"role": "user", "content": user_message})
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return response, rag_metadata
    
    def reset_conversation(self):
        self.conversation_history = []

# =============================================================================
# BUILD KNOWLEDGE BASE
# =============================================================================

print("="*70)
print("  🌟 SERENITY - Building Knowledge Base")
print("="*70)

# Scrape articles
print("\n[1/3] Scraping articles...")
scraper = KnowledgeScraper()
urls = [
    "https://en.wikipedia.org/wiki/Empathy",
    "https://en.wikipedia.org/wiki/Emotional_intelligence",
    "https://en.wikipedia.org/wiki/Active_listening",
    "https://en.wikipedia.org/wiki/Cognitive_behavioral_therapy",
    "https://en.wikipedia.org/wiki/Mindfulness",
    "https://en.wikipedia.org/wiki/Compassion",
    "https://en.wikipedia.org/wiki/Coping",
]
articles = scraper.scrape_urls(urls, delay=1.5)
print(f"✓ Scraped {len(articles)} articles")

# Process into chunks
print("\n[2/3] Processing chunks...")
processor = TextProcessor(chunk_size=180, overlap=30)
chunks = processor.process_articles(articles)
print(f"✓ Created {len(chunks)} chunks")

# Build index
print("\n[3/3] Building FAISS index...")
engine = EmbeddingEngine()
engine.build_index(chunks)

# Initialize Serenity
print("\nInitializing Serenity (downloading model ~3GB, first time only)...")
serenity = SerenityGenerator(embedding_engine=engine)

print("\n" + "="*70)
print("  ✅ READY! Serenity is loaded and ready to chat!")
print("="*70)

# =============================================================================
# CHAT FUNCTION
# =============================================================================

def chat():
    """Interactive chat with Serenity"""
    print("\n🌟 SERENITY CHAT")
    print("Commands: 'quit' to exit | 'clear' to reset | 'sources' to toggle\n")
    
    show_sources = False
    
    while True:
        user_input = input("You: ").strip()
        
        if not user_input:
            continue
            
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Take care! 💙\n")
            break
            
        if user_input.lower() == 'clear':
            serenity.reset_conversation()
            print("\n✓ Conversation cleared\n")
            continue
            
        if user_input.lower() == 'sources':
            show_sources = not show_sources
            print(f"\n✓ Sources {'ON' if show_sources else 'OFF'}\n")
            continue
        
        try:
            response, metadata = serenity.generate_response(user_input)
            
            if not response or response.strip() == "":
                print("\nSerenity: I'm here to listen. Could you tell me more about what's on your mind?\n")
            else:
                print(f"\nSerenity: {response}\n")
            
            if show_sources and metadata:
                print("📚 Sources:", [m['source'] for m in metadata], "\n")
                
        except Exception as e:
            print(f"\n❌ Error: {e}\n")
            import traceback
            traceback.print_exc()

# =============================================================================
# AUTO-START CHAT
# =============================================================================

print("\n🎯 Starting chat automatically...\n")
chat()

In [5]:
"""
SERENITY - RAG Therapist (Concise & Empathetic)
Copy this ENTIRE file into ONE cell and run it
"""

# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================
!pip install -q beautifulsoup4 requests sentence-transformers faiss-cpu transformers accelerate

# =============================================================================
# IMPORTS
# =============================================================================
import requests
from bs4 import BeautifulSoup
import time, os, re
from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =============================================================================
# DATA STRUCTURES
# =============================================================================
@dataclass
class Chunk:
    chunk_id: str
    text: str
    source_title: str
    source_url: str
    source_type: str
    word_count: int
    chunk_index: int
    def to_dict(self) -> Dict:
        return asdict(self)

# =============================================================================
# SCRAPER
# =============================================================================
class KnowledgeScraper:
    def __init__(self):
        self.headers = {'User-Agent': 'Mozilla/5.0'}
    def scrape_wikipedia_article(self, url: str) -> Dict:
        try:
            r = requests.get(url, headers=self.headers, timeout=10)
            r.raise_for_status()
            soup = BeautifulSoup(r.content, 'html.parser')
            title = soup.find('h1', class_='firstHeading').get_text() if soup.find('h1', class_='firstHeading') else "Unknown"
            content_div = soup.find('div', class_='mw-parser-output')
            if not content_div: return None
            paragraphs = []
            for p in content_div.find_all('p'):
                text = p.get_text().strip()
                if len(text) > 40 and not text.startswith('['):
                    text = re.sub(r'\[\d+\]', '', text)
                    paragraphs.append(text)
            content = "\n\n".join(paragraphs)
            return {'title': title, 'url': url, 'content': content, 'source': 'Wikipedia', 'word_count': len(content.split())}
        except:
            return None
    def scrape_urls(self, urls: List[str], delay: float = 1.0) -> List[Dict]:
        articles = []
        for i, url in enumerate(urls):
            print(f"Scraping {i+1}/{len(urls)}: {url}")
            article = self.scrape_wikipedia_article(url)
            if article and article['word_count'] > 50:
                articles.append(article)
                print(f"  ✓ {article['title']} ({article['word_count']} words)")
            if i < len(urls) - 1: time.sleep(delay)
        return articles

# =============================================================================
# TEXT PROCESSOR
# =============================================================================
class TextProcessor:
    def __init__(self, chunk_size=180, overlap=30):
        self.chunk_size = chunk_size
        self.overlap = overlap
    def clean_text(self, text: str) -> str:
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\w\s.,!?;:\-\'"()]', '', text)
        return text.strip()
    def split_into_sentences(self, text: str) -> List[str]:
        return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    def create_chunks(self, text: str) -> List[str]:
        sentences = self.split_into_sentences(text)
        chunks, current_chunk, current_words = [], [], 0
        for s in sentences:
            wc = len(s.split())
            if current_words + wc > self.chunk_size and current_chunk:
                chunks.append(' '.join(current_chunk))
                overlap_sentences, overlap_words = [], 0
                for sent in reversed(current_chunk):
                    if overlap_words + len(sent.split()) <= self.overlap:
                        overlap_sentences.insert(0, sent)
                        overlap_words += len(sent.split())
                    else: break
                current_chunk, current_words = overlap_sentences, overlap_words
            current_chunk.append(s)
            current_words += wc
        if current_chunk: chunks.append(' '.join(current_chunk))
        return chunks
    def process_articles(self, articles: List[Dict]) -> List[Chunk]:
        all_chunks = []
        for idx, article in enumerate(articles):
            cleaned = self.clean_text(article['content'])
            chunk_texts = self.create_chunks(cleaned)
            for c_idx, text in enumerate(chunk_texts):
                all_chunks.append(Chunk(
                    chunk_id=f"article_{idx}_chunk_{c_idx}",
                    text=text,
                    source_title=article['title'],
                    source_url=article['url'],
                    source_type=article['source'],
                    word_count=len(text.split()),
                    chunk_index=c_idx
                ))
            print(f"Processed: {article['title']} -> {len(chunk_texts)} chunks")
        return all_chunks

# =============================================================================
# EMBEDDING ENGINE
# =============================================================================
class EmbeddingEngine:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        self.index = None
        self.chunks = []
    def build_index(self, chunks: List[Chunk]):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        embeddings = self.model.encode(texts, batch_size=32, convert_to_numpy=True)
        faiss.normalize_L2(embeddings)
        self.index = faiss.IndexFlatIP(self.embedding_dim)
        self.index.add(embeddings.astype('float32'))
        print(f"✓ FAISS index built with {self.index.ntotal} vectors")
    def get_context(self, query: str, top_k=3, max_words=300) -> Tuple[str, List[Dict]]:
        q_emb = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(q_emb)
        sims, idxs = self.index.search(q_emb.astype('float32'), min(top_k, self.index.ntotal))
        context, metadata, total_words = [], [], 0
        for i, (idx, score) in enumerate(zip(idxs[0], sims[0])):
            chunk = self.chunks[idx]
            if total_words + chunk.word_count > max_words and i > 0: break
            context.append(f"[Source {i+1}]: {chunk.text}")
            metadata.append({'source': chunk.source_title, 'url': chunk.source_url, 'score': float(score)})
            total_words += chunk.word_count
        return "\n\n".join(context), metadata

# =============================================================================
# SERENITY LLM
# =============================================================================
class SerenityGenerator:
    def __init__(self, model_name="Qwen/Qwen2.5-1.5B-Instruct", embedding_engine=None):
        print(f"Loading model: {model_name}...")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, torch_dtype=torch.float16)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None: self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.eval()
        self.embedding_engine = embedding_engine
        self.conversation_history = []
        print("✓ Serenity loaded!")
    def get_system_prompt(self):
        return """You are Serenity, a warm and empathetic therapist. Respond CONCISELY (2-3 sentences), validate feelings, and reflect. Do NOT give long lists or step-by-step advice. Keep responses human-like and compassionate."""
    def generate_response(self, user_message: str) -> Tuple[str, List[Dict]]:
        context, metadata = ("", [])
        if self.embedding_engine:
            context, metadata = self.embedding_engine.get_context(user_message)
        system_prompt = self.get_system_prompt()
        if context: system_prompt += f"\nContext:\n{context}"
        # Build input text
        input_text = f"{system_prompt}\nConversation:\n"
        for msg in self.conversation_history[-6:]:
            role = msg['role']
            input_text += f"{role.capitalize()}: {msg['content']}\n"
        input_text += f"User: {user_message}\nAssistant:"
        # Tokenize
        inputs = self.tokenizer([input_text], return_tensors="pt").to(self.model.device)
        # Generate
        outputs = self.model.generate(**inputs, max_new_tokens=150, temperature=0.7, top_p=0.9, do_sample=True, pad_token_id=self.tokenizer.pad_token_id)
        response = self.tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
        # Update history
        self.conversation_history.append({"role":"user","content":user_message})
        self.conversation_history.append({"role":"assistant","content":response})
        return response, metadata
    def reset_conversation(self): self.conversation_history = []

# =============================================================================
# BUILD KNOWLEDGE BASE
# =============================================================================
scraper = KnowledgeScraper()
urls = [
    "https://en.wikipedia.org/wiki/Empathy",
    "https://en.wikipedia.org/wiki/Emotional_intelligence",
    "https://en.wikipedia.org/wiki/Active_listening",
    "https://en.wikipedia.org/wiki/Cognitive_behavioral_therapy",
    "https://en.wikipedia.org/wiki/Mindfulness",
    "https://en.wikipedia.org/wiki/Compassion",
    "https://en.wikipedia.org/wiki/Coping",
]
articles = scraper.scrape_urls(urls)
processor = TextProcessor()
chunks = processor.process_articles(articles)
engine = EmbeddingEngine()
engine.build_index(chunks)
serenity = SerenityGenerator(embedding_engine=engine)

# =============================================================================
# CHAT LOOP
# =============================================================================
def chat():
    print("\n🌿 SERENITY CHAT (Concise Therapist Style)")
    print("Type 'quit' to exit | 'clear' to reset conversation\n")
    while True:
        user_input = input("You: ").strip()
        if not user_input: continue
        if user_input.lower() in ['quit', 'exit']: break
        if user_input.lower() == 'clear':
            serenity.reset_conversation()
            print("✓ Conversation cleared")
            continue
        try:
            response, metadata = serenity.generate_response(user_input)
            print(f"Serenity: {response}\n")
        except Exception as e:
            print(f"Error: {e}\n")

print("\n🎯 Starting concise therapist chat...\n")
chat()


2026-02-16 17:30:37.781483: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771263037.940734      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771263037.987127      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771263038.391608      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771263038.391655      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771263038.391658      55 computation_placer.cc:177] computation placer alr

Scraping 1/7: https://en.wikipedia.org/wiki/Empathy
  ✓ Empathy (10266 words)
Scraping 2/7: https://en.wikipedia.org/wiki/Emotional_intelligence
  ✓ Emotional intelligence (4129 words)
Scraping 3/7: https://en.wikipedia.org/wiki/Active_listening
  ✓ Active listening (3157 words)
Scraping 4/7: https://en.wikipedia.org/wiki/Cognitive_behavioral_therapy
  ✓ Cognitive behavioral therapy (8399 words)
Scraping 5/7: https://en.wikipedia.org/wiki/Mindfulness
  ✓ Mindfulness (8346 words)
Scraping 6/7: https://en.wikipedia.org/wiki/Compassion
  ✓ Compassion (4645 words)
Scraping 7/7: https://en.wikipedia.org/wiki/Coping
  ✓ Coping (2477 words)
Processed: Empathy -> 69 chunks
Processed: Emotional intelligence -> 27 chunks
Processed: Active listening -> 21 chunks
Processed: Cognitive behavioral therapy -> 56 chunks
Processed: Mindfulness -> 58 chunks
Processed: Compassion -> 33 chunks
Processed: Coping -> 16 chunks
Loading embedding model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Serenity loaded!

🎯 Starting concise therapist chat...


🌿 SERENITY CHAT (Concise Therapist Style)
Type 'quit' to exit | 'clear' to reset conversation



You:  hi


Serenity: Hello there! How can I assist you today? 😊

Do you feel like sharing something on how you've been feeling recently? I'm here to listen and support you, no matter what you're going through. Remember, your feelings are valid, and it's okay to express them. Together we can work towards understanding and finding ways to move forward. 🌟💖

Remember, sometimes just knowing you're heard makes all the difference. You're not alone in this journey. Let's talk about whatever brings you comfort right now. 📝💬

Feel free to share anything that comes to mind. Your honesty matters to me. ❤️💕

Is there anything specific you'd like to discuss further? Please let me know.



You:  quit


In [6]:
"""
SERENITY - RAG Therapist (SHORT EMPATHETIC RESPONSES)
Copy this ENTIRE file into ONE cell and run it
"""

# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================
!pip install -q beautifulsoup4 requests sentence-transformers faiss-cpu transformers accelerate

# =============================================================================
# IMPORTS
# =============================================================================
import requests
from bs4 import BeautifulSoup
import time
import json
import os
from typing import List, Dict, Tuple
from urllib.parse import urlparse
import re
from dataclasses import dataclass, asdict
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =============================================================================
# CLASSES
# =============================================================================

@dataclass
class Chunk:
    chunk_id: str
    text: str
    source_title: str
    source_url: str
    source_type: str
    word_count: int
    chunk_index: int
    def to_dict(self) -> Dict:
        return asdict(self)

class KnowledgeScraper:
    def __init__(self, output_dir: str = "knowledge_base"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        
    def scrape_wikipedia_article(self, url: str) -> Dict:
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            title = soup.find('h1', class_='firstHeading')
            title_text = title.get_text() if title else "Unknown"
            content_div = soup.find('div', class_='mw-parser-output')
            if not content_div:
                return None
            paragraphs = []
            for p in content_div.find_all('p'):
                text = p.get_text().strip()
                if len(text) > 40 and not text.startswith('['):
                    text = re.sub(r'\[\d+\]', '', text)
                    paragraphs.append(text)
            content = '\n\n'.join(paragraphs)
            return {'title': title_text, 'url': url, 'content': content, 'source': 'Wikipedia', 'word_count': len(content.split())}
        except Exception as e:
            print(f"Error scraping {url}: {str(e)}")
            return None
    
    def scrape_urls(self, urls: List[str], delay: float = 1.0) -> List[Dict]:
        articles = []
        for i, url in enumerate(urls):
            print(f"Scraping {i+1}/{len(urls)}: {url}")
            article = self.scrape_wikipedia_article(url)
            if article and article['word_count'] > 50:
                articles.append(article)
                print(f"  ✓ {article['title']} ({article['word_count']} words)")
            if i < len(urls) - 1:
                time.sleep(delay)
        return articles

class TextProcessor:
    def __init__(self, chunk_size: int = 180, overlap: int = 30):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def clean_text(self, text: str) -> str:
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\w\s.,!?;:\-\'"()]', '', text)
        return text.strip()
    
    def split_into_sentences(self, text: str) -> List[str]:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sentences if s.strip()]
    
    def create_chunks(self, text: str, min_words: int = 40) -> List[str]:
        sentences = self.split_into_sentences(text)
        chunks = []
        current_chunk = []
        current_word_count = 0
        for sentence in sentences:
            sentence_word_count = len(sentence.split())
            if current_word_count + sentence_word_count > self.chunk_size and current_chunk:
                chunks.append(' '.join(current_chunk))
                overlap_sentences = []
                overlap_words = 0
                for sent in reversed(current_chunk):
                    if overlap_words + len(sent.split()) <= self.overlap:
                        overlap_sentences.insert(0, sent)
                        overlap_words += len(sent.split())
                    else:
                        break
                current_chunk = overlap_sentences
                current_word_count = overlap_words
            current_chunk.append(sentence)
            current_word_count += sentence_word_count
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        return chunks
    
    def process_article(self, article: Dict, article_idx: int) -> List[Chunk]:
        cleaned_content = self.clean_text(article['content'])
        chunk_texts = self.create_chunks(cleaned_content)
        chunks = []
        for chunk_idx, chunk_text in enumerate(chunk_texts):
            chunk = Chunk(
                chunk_id=f"article_{article_idx}_chunk_{chunk_idx}",
                text=chunk_text,
                source_title=article['title'],
                source_url=article['url'],
                source_type=article['source'],
                word_count=len(chunk_text.split()),
                chunk_index=chunk_idx
            )
            chunks.append(chunk)
        return chunks
    
    def process_articles(self, articles: List[Dict]) -> List[Chunk]:
        all_chunks = []
        for idx, article in enumerate(articles):
            chunks = self.process_article(article, idx)
            all_chunks.extend(chunks)
            print(f"Processed: {article['title']} -> {len(chunks)} chunks")
        return all_chunks

class EmbeddingEngine:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        self.index = None
        self.chunks = []
        
    def build_index(self, chunks: List[Chunk]):
        print(f"Building FAISS index for {len(chunks)} chunks...")
        self.chunks = chunks
        chunk_texts = [chunk.text for chunk in chunks]
        embeddings = self.model.encode(chunk_texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
        faiss.normalize_L2(embeddings)
        self.index = faiss.IndexFlatIP(self.embedding_dim)
        self.index.add(embeddings.astype('float32'))
        print(f"✓ FAISS index built with {self.index.ntotal} vectors")
    
    def search(self, query: str, top_k: int = 3) -> List[Tuple[Chunk, float]]:
        query_embedding = self.model.encode([query], show_progress_bar=False, convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)
        similarities, indices = self.index.search(query_embedding.astype('float32'), min(top_k, self.index.ntotal))
        results = []
        for idx, score in zip(indices[0], similarities[0]):
            if idx < len(self.chunks):
                results.append((self.chunks[idx], float(score)))
        return results
    
    def get_context(self, query: str, top_k: int = 3, max_words: int = 500) -> Tuple[str, List[Dict]]:
        results = self.search(query, top_k)
        context_parts = []
        metadata = []
        total_words = 0
        for i, (chunk, score) in enumerate(results):
            if total_words + chunk.word_count > max_words and i > 0:
                break
            context_parts.append(f"[Source {i+1}]: {chunk.text}")
            total_words += chunk.word_count
            metadata.append({'source': chunk.source_title, 'url': chunk.source_url, 'similarity': score, 'chunk_id': chunk.chunk_id})
        return "\n\n".join(context_parts), metadata

class SerenityGenerator:
    def __init__(self, base_model: str = "Qwen/Qwen2.5-1.5B-Instruct", embedding_engine=None):
        print(f"Loading model: {base_model}...")
        self.model = AutoModelForCausalLM.from_pretrained(
            base_model, 
            device_map="auto", 
            trust_remote_code=True, 
            torch_dtype=torch.float16
        )
        self.tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.eval()
        self.embedding_engine = embedding_engine
        self.conversation_history = []
        print("✓ Serenity loaded!")
    
    def get_system_prompt(self) -> str:
        return """You are Serenity, a warm empathetic therapist. 

CRITICAL RULES:
- Keep responses SHORT (2-4 sentences, max 60 words)
- Be conversational and warm, not formal
- Reflect feelings, then ask ONE follow-up question
- NO bullet points, NO lists, NO numbered steps
- Talk like a real therapist in a gentle conversation
- Use the knowledge context subtly, don't lecture

Examples of good responses:
"I hear you - breakups are really painful. What's been the hardest part for you?"
"That sounds overwhelming. When did you first start noticing these feelings?"
"It makes sense you'd feel anxious about that. Have you been able to talk to anyone about this?"

Remember: Brief, empathetic, conversational. One thoughtful question."""
    
    def generate_response(self, user_message: str, use_rag: bool = True, top_k_chunks: int = 2, max_new_tokens: int = 80, temperature: float = 0.7, top_p: float = 0.9) -> Tuple[str, List[Dict]]:
        # Get context
        context = ""
        rag_metadata = []
        if use_rag and self.embedding_engine:
            context, rag_metadata = self.embedding_engine.get_context(user_message, top_k=top_k_chunks, max_words=250)
        
        # Build messages
        messages = []
        system_content = self.get_system_prompt()
        if context:
            system_content += f"\n\nRelevant context (use subtly):\n{context}"
        messages.append({"role": "system", "content": system_content})
        
        # Add history
        if self.conversation_history:
            for msg in self.conversation_history[-4:]:  # Less history for shorter responses
                messages.append(msg)
        
        # Add current message
        messages.append({"role": "user", "content": user_message})
        
        # Generate
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,  # Shorter responses
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id
        )
        
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Clean up response - remove any bullet points or lists
        response = re.sub(r'\n\d+\..*', '', response)  # Remove numbered lists
        response = re.sub(r'\n-.*', '', response)  # Remove bullet points
        response = re.sub(r'\*\*.*?\*\*', '', response)  # Remove bold
        response = response.strip()
        
        # Update history
        self.conversation_history.append({"role": "user", "content": user_message})
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return response, rag_metadata
    
    def reset_conversation(self):
        self.conversation_history = []

# =============================================================================
# BUILD KNOWLEDGE BASE
# =============================================================================

print("="*70)
print("  🌟 SERENITY - Building Knowledge Base")
print("="*70)

# Scrape articles
print("\n[1/3] Scraping articles...")
scraper = KnowledgeScraper()
urls = [
    "https://en.wikipedia.org/wiki/Empathy",
    "https://en.wikipedia.org/wiki/Emotional_intelligence",
    "https://en.wikipedia.org/wiki/Active_listening",
    "https://en.wikipedia.org/wiki/Cognitive_behavioral_therapy",
    "https://en.wikipedia.org/wiki/Mindfulness",
    "https://en.wikipedia.org/wiki/Compassion",
    "https://en.wikipedia.org/wiki/Coping",
]
articles = scraper.scrape_urls(urls, delay=1.5)
print(f"✓ Scraped {len(articles)} articles")

# Process into chunks
print("\n[2/3] Processing chunks...")
processor = TextProcessor(chunk_size=180, overlap=30)
chunks = processor.process_articles(articles)
print(f"✓ Created {len(chunks)} chunks")

# Build index
print("\n[3/3] Building FAISS index...")
engine = EmbeddingEngine()
engine.build_index(chunks)

# Initialize Serenity
print("\nInitializing Serenity (downloading model ~3GB, first time only)...")
serenity = SerenityGenerator(embedding_engine=engine)

print("\n" + "="*70)
print("  ✅ READY! Serenity is loaded and ready to chat!")
print("="*70)

# =============================================================================
# CHAT FUNCTION
# =============================================================================

def chat():
    """Interactive chat with Serenity"""
    print("\n🌟 SERENITY CHAT - Short & Empathetic Responses")
    print("Commands: 'quit' to exit | 'clear' to reset | 'sources' to toggle\n")
    
    show_sources = False
    
    while True:
        user_input = input("You: ").strip()
        
        if not user_input:
            continue
            
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Take care! 💙\n")
            break
            
        if user_input.lower() == 'clear':
            serenity.reset_conversation()
            print("\n✓ Conversation cleared\n")
            continue
            
        if user_input.lower() == 'sources':
            show_sources = not show_sources
            print(f"\n✓ Sources {'ON' if show_sources else 'OFF'}\n")
            continue
        
        try:
            response, metadata = serenity.generate_response(user_input)
            
            if not response or response.strip() == "":
                print("\nSerenity: I'm here to listen. What's on your mind?\n")
            else:
                print(f"\nSerenity: {response}\n")
            
            if show_sources and metadata:
                print("📚 Sources:", [m['source'] for m in metadata], "\n")
                
        except Exception as e:
            print(f"\n❌ Error: {e}\n")
            import traceback
            traceback.print_exc()

# =============================================================================
# AUTO-START CHAT
# =============================================================================

print("\n🎯 Starting chat automatically...\n")
chat()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  🌟 SERENITY - Building Knowledge Base

[1/3] Scraping articles...
Scraping 1/7: https://en.wikipedia.org/wiki/Empathy
  ✓ Empathy (10266 words)
Scraping 2/7: https://en.wikipedia.org/wiki/Emotional_intelligence
  ✓ Emotional intelligence (4129 words)
Scraping 3/7: https://en.wikipedia.org/wiki/Active_listening
  ✓ Active listening (3157 words)
Scraping 4/7: https://en.wikipedia.org/wiki/Cognitive_behavioral_therapy
  ✓ Cognitive behavioral therapy (8399 words)
Scraping 5/7: https://en.wikipedia.org/wiki/Mindfulness
  ✓ Mindfulness (8346 words)
Scraping 6/7: https://en.wikipedia.org/wiki/Compassion
  ✓ Compassion (4645 words)
Scraping 7/7: https://en.wikipedia.org/wiki/Coping
  ✓ Coping (2477 words)
✓ Scraped 7 articles

[2/3] Processing chunks...
Processed: Empathy -> 69 chunks
Processed: Emotional intelligence -> 27 chunks
Processed: Active listening -> 21 chunks
Processed: Cognitive behavioral therapy -> 56 chunks
Processed: Mindfulness -> 58 chunks
Processed: Compassion -> 33 chunk

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

✓ FAISS index built with 280 vectors

Initializing Serenity (downloading model ~3GB, first time only)...
Loading model: Qwen/Qwen2.5-1.5B-Instruct...
✓ Serenity loaded!

  ✅ READY! Serenity is loaded and ready to chat!

🎯 Starting chat automatically...


🌟 SERENITY CHAT - Short & Empathetic Responses
Commands: 'quit' to exit | 'clear' to reset | 'sources' to toggle



You:  hi



Serenity: Hello there! How can I assist you today?



You:  quit



👋 Take care! 💙



In [3]:
import faiss
import pickle

# -----------------------------
# Paths for saving
# -----------------------------
model_save_path = "./qwen2_5_local"
faiss_index_path = "serenity_faiss.index"
chunks_path = "serenity_chunks.pkl"

# -----------------------------
# Save model & tokenizer from Serenity
# -----------------------------
serenity.model.save_pretrained(model_save_path)
serenity.tokenizer.save_pretrained(model_save_path)
print("✓ Model and tokenizer saved locally!")

# -----------------------------
# Save FAISS index
# -----------------------------
faiss.write_index(serenity.embedding_engine.index, faiss_index_path)
print(f"✓ FAISS index saved to {faiss_index_path}")

# -----------------------------
# Save chunks metadata
# -----------------------------
with open(chunks_path, "wb") as f:
    pickle.dump(serenity.embedding_engine.chunks, f)
print(f"✓ Chunks saved to {chunks_path}")


✓ Model and tokenizer saved locally!
✓ FAISS index saved to serenity_faiss.index
✓ Chunks saved to serenity_chunks.pkl


In [2]:
import faiss
import pickle
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# Paths for loading
# -----------------------------
model_load_path = "./qwen2_5_local"
faiss_index_path = "serenity_faiss.index"
chunks_path = "serenity_chunks.pkl"

# -----------------------------
# Load FAISS index
# -----------------------------
index = faiss.read_index(faiss_index_path)
print(f"✓ FAISS index loaded from {faiss_index_path}")

# -----------------------------
# Load chunks
# -----------------------------
with open(chunks_path, "rb") as f:
    chunks = pickle.load(f)
print(f"✓ Loaded {len(chunks)} chunks")

# -----------------------------
# Define embedding engine
# -----------------------------
from sentence_transformers import SentenceTransformer
class EmbeddingEngine:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        self.index = None
        self.chunks = []

embedding_engine = EmbeddingEngine()
embedding_engine.index = index
embedding_engine.chunks = chunks

# -----------------------------
# Load model & tokenizer
# -----------------------------
model = AutoModelForCausalLM.from_pretrained(model_load_path, device_map="auto", torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_load_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✓ Model and tokenizer loaded successfully!")

# -----------------------------
# Recreate Serenity generator
# -----------------------------
class SerenityGenerator:
    def __init__(self, model, tokenizer, embedding_engine):
        self.model = model
        self.tokenizer = tokenizer
        self.embedding_engine = embedding_engine
        self.conversation_history = []

    def get_system_prompt(self):
        return """You are Serenity, a warm and empathetic therapist assistant. 
Listen actively, validate feelings, and respond concisely and human-like."""

    def generate_response(self, user_message: str, use_rag=True, top_k_chunks=3, max_new_tokens=250):
        # RAG context
        context = ""
        if use_rag and self.embedding_engine:
            # get top-k chunks
            results = self.embedding_engine.index.search(
                self.embedding_engine.model.encode([user_message], convert_to_numpy=True).astype('float32'), 
                min(top_k_chunks, self.embedding_engine.index.ntotal)
            )
            # For simplicity, just combine top texts
            context = " ".join([self.embedding_engine.chunks[i].text for i in results[1][0] if i < len(self.embedding_engine.chunks)])
        
        system_prompt = self.get_system_prompt()
        if context:
            system_prompt += f"\n\nContext:\n{context}"

        messages = [{"role": "system", "content": system_prompt}]
        messages.append({"role": "user", "content": user_message})
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.conversation_history.append({"role": "user", "content": user_message})
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return response

# -----------------------------
# Instantiate Serenity
# -----------------------------
serenity = SerenityGenerator(model, tokenizer, embedding_engine)
print("🌿 Serenity loaded and ready to chat!")

# -----------------------------
# Simple chat loop
# -----------------------------
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["quit", "exit"]:
        print("👋 Goodbye!")
        break
    response = serenity.generate_response(user_input)
    print(f"Serenity: {response}\n")


ModuleNotFoundError: No module named 'faiss'

In [ ]:
import shutil
import os

# Create a temporary folder for just the essentials
os.makedirs('/kaggle/working/serenity_essentials', exist_ok=True)

# List of essential files to copy (based on your screenshot)
essentials = [
    'serenity_chunks.pkl',
    'serenity_faiss.index',
    'knowledge_base'
]

for item in essentials:
    src = os.path.join('/kaggle/working/', item)
    dst = os.path.join('/kaggle/working/serenity_essentials/', item)
    if os.path.exists(src):
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)

# Zip only the essentials
shutil.make_archive('serenity_minimal', 'zip', '/kaggle/working/serenity_essentials')
print("✅ Created 'serenity_minimal.zip'. You can download this safely now!")

In [4]:
import os, shutil

package = "/kaggle/working/serenity_offline"
os.makedirs(package, exist_ok=True)

# copy model
shutil.copytree("/kaggle/working/qwen2_5_local",
                f"{package}/model")

# copy rag
shutil.copy("/kaggle/working/serenity_faiss.index",
            f"{package}/serenity_faiss.index")

shutil.copy("/kaggle/working/serenity_chunks.pkl",
            f"{package}/serenity_chunks.pkl")

print("Package ready")


Package ready


In [5]:
!zip -r /kaggle/working/serenity_offline.zip /kaggle/working/serenity_offline



  adding: kaggle/working/serenity_offline/ (stored 0%)
  adding: kaggle/working/serenity_offline/serenity_chunks.pkl (deflated 69%)
  adding: kaggle/working/serenity_offline/serenity_faiss.index (deflated 7%)
  adding: kaggle/working/serenity_offline/model/ (stored 0%)
  adding: kaggle/working/serenity_offline/model/added_tokens.json (deflated 67%)
  adding: kaggle/working/serenity_offline/model/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 81%)
  adding: kaggle/working/serenity_offline/model/config.json (deflated 71%)
  adding: kaggle/working/serenity_offline/model/special_tokens_map.json (deflated 69%)
  adding: kaggle/working/serenity_offline/model/tokenizer_config.json (deflated 89%)
  adding: kaggle/working/serenity_offline/model/chat_template.jinja (deflated 71%)
  adding: kaggle/working/serenity_offline/model/merges.txt (deflated 57%)
  adding: kaggle/working/serenity_offline/model/model.safetensors (deflated 23%)
  adding: kaggle/working/serenity_offline/model/generation_config.json (deflated 39%)
  adding: kaggle/working/serenity_offline/model/vocab.json (deflated 61%)


In [9]:
from IPython.display import FileLink
FileLink(r'qwen2_5_quantized.zip')

/kaggle/working/qwen2_5_quantized.zip

In [7]:
# ============================================================
# 0) Install dependencies
# ============================================================
!pip -q install transformers accelerate bitsandbytes safetensors

# ============================================================
# 1) Imports
# ============================================================
import torch
import os
import shutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ============================================================
# 2) Settings
# ============================================================
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SAVE_DIR   = "/kaggle/working/qwen2_5_quantized"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Using GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 3) Quantization config (4-bit NF4 — best quality/speed balance)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# ============================================================
# 4) Load model in quantized mode
# ============================================================
print("\nLoading base model and quantizing...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded in 4-bit")

# ============================================================
# 5) Save quantized model (portable format)
# ============================================================
print("\nSaving quantized model...")

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved at:", SAVE_DIR)

# ============================================================
# 6) Verify files
# ============================================================
print("\nSaved files:")
for f in os.listdir(SAVE_DIR):
    print(" -", f)

# ============================================================
# 7) Zip for download
# ============================================================
print("\nCreating zip...")

zip_path = "/kaggle/working/qwen2_5_quantized"
shutil.make_archive(zip_path, 'zip', SAVE_DIR)

print("\nDONE ✅ Download file:")
print(zip_path + ".zip")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Using GPU: Tesla T4

Loading base model and quantizing...
Model loaded in 4-bit

Saving quantized model...
Saved at: /kaggle/working/qwen2_5_quantized

Saved files:
 - tokenizer.json
 - merges.txt
 - added_tokens.json
 - special_tokens_map.json
 - config.json
 - generation_config.json
 - tokenizer_config.json
 - vocab.json
 - model.safetensors
 - chat_template.jinja

Creating zip...

DONE ✅ Download file:
/kaggle/working/qwen2_5_quantized.zip
